# Learning Rate Finder

Quick implementation of the LR range test (Smith 2017): warm up the LR exponentially while training, plot loss vs LR, pick the LR where loss is still descending steeply but well before divergence.

In [ ]:
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

In [ ]:
def lr_range_test(model, loader, criterion, optimizer, lr_start=1e-7, lr_end=10, num_iter=100):
    lrs = []
    losses = []
    multiplier = (lr_end / lr_start) ** (1 / num_iter)
    lr = lr_start
    for g in optimizer.param_groups:
        g['lr'] = lr
    model.train()
    avg_loss = 0.0
    beta = 0.98
    for i, (x, y) in enumerate(loader):
        if i >= num_iter:
            break
        out = model(x)
        loss = criterion(out, y)
        avg_loss = beta * avg_loss + (1 - beta) * loss.item()
        smoothed = avg_loss / (1 - beta ** (i + 1))
        if i > 0 and smoothed > 4 * min(losses):
            break
        lrs.append(lr)
        losses.append(smoothed)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr *= multiplier
        for g in optimizer.param_groups:
            g['lr'] = lr
    return lrs, losses

In [ ]:
# Synthetic example: tiny MLP on random data
torch.manual_seed(42)
X = torch.randn(2048, 32)
y = (X.sum(dim=1) > 0).long()
loader = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)
model = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 2))
optimizer = torch.optim.SGD(model.parameters(), lr=1e-7, momentum=0.9)
criterion = nn.CrossEntropyLoss()
lrs, losses = lr_range_test(model, loader, criterion, optimizer)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lrs, losses)
ax.set_xscale('log')
ax.set_xlabel('learning rate')
ax.set_ylabel('loss')
ax.set_title('LR range test')
ax.grid(alpha=0.3)
plt.show()

## How to read this

Pick an LR ~one decade BEFORE the loss starts diverging. For most networks this lands somewhere between 1e-3 and 1e-2.